# CSE465 ColorBench - Adaptive Skill Generation Pipeline

**Architecture:** Agentic Context Engineering (ACE Framework - ICLR 2026)
- **Self-Improving Agent:** Qwen2.5-VL-7B (4-bit NF4) acts as Generator, Reflector, and Solver.
- **Phase 1 (Skill Generator):** Generates visually-grounded cognitive skills.
- **Phase 2 (Skill Reflector):** Reflects on prediction errors and iteratively refines the playbook.
- **Phase 3 (Visual Solver):** Solves the question with the exact mandated prompt phrasing:
  *"This is the skill to solve this question, now solve the question and give me the answer."*

**Target:** Google Colab Free-Tier (Tesla T4 GPU, 15GB VRAM) — 100% Local Inference (0 API Keys)


## 1. Mount Google Drive


In [ ]:
from google.colab import drive
import os
from datetime import datetime

# Mount Drive
drive.mount('/content/drive')

# Set up Run Tag and Directory
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S") + "_ACE_Pipeline"
DRIVE_SAVE_DIR = f"/content/drive/MyDrive/CSE465_Results/{RUN_TAG}"

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"\nAll results will be saved to Google Drive: {DRIVE_SAVE_DIR}")


## 2. Install Dependencies


In [ ]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils datasets "pillow<11.0.0"


## 3. Clone Repository


In [ ]:
import os

REPO_URL = "https://github.com/YOUR_USERNAME/cse465-project.git"  # <-- UPDATE THIS

# Go back to /content before cloning
%cd /content
if not os.path.exists("cse465-project"):
    !git clone {REPO_URL}

%cd cse465-project


## 4. Verify GPU & API Key


In [ ]:
import torch

# GPU Verification
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {name} ({vram:.1f} GB VRAM) — Ready for 100% Local Execution.")
else:
    print("WARNING: No GPU detected! Set runtime type to GPU (T4).")


## 5. Load Models (Run Once)

Loads Qwen2.5-VL-7B in 4-bit precision (takes ~2 min).


In [ ]:
from qwen_solver import QwenSolver
from ace import ACE

# Load Qwen2.5-VL-7B in 4-bit (Unified Generator + Reflector + Solver)
solver = QwenSolver()
solver.load_model()

# Initialize ACE with the local solver (Self-Improving Agent)
ace_system = ACE(solver=solver)

print("\nAll models ready! Running 100% locally on Colab GPU.")


## 6. Load Dataset (Run Once)


In [ ]:
from data_loader import ColorBenchDataLoader

loader = ColorBenchDataLoader(task_filter="Color Recognition")
dataset_items = list(loader.stream_instances())
print(f"Loaded {len(dataset_items)} Color Recognition questions.")


## 7. Experiment: Baseline (Direct VQA, No Skills)


In [ ]:
import time
import os
import json
from data_loader import IncrementalLogger

# Save directly to Google Drive
baseline_path = os.path.join(DRIVE_SAVE_DIR, "results_color_recognition_baseline.jsonl")
baseline_logger = IncrementalLogger(baseline_path)

# TEST MODE: Set to False to run all 76 questions
QUICK_TEST = True
limit = 3 if QUICK_TEST else len(dataset_items)

correct, total = 0, 0
start = time.time()

for item in dataset_items[:limit]:
    if item["idx"] in baseline_logger.processed_indices:
        continue

    result = solver.solve(item["image"], item["question"], item["choices"], mode="baseline")
    pred = result["prediction"]
    gt = item["answer"]
    is_correct = (pred == gt)
    if is_correct: correct += 1
    total += 1

    baseline_logger.log_result({
        "idx": item["idx"], "id": item["id"], "task": item["task"],
        "question": item["question"], "choices": item["choices"],
        "ground_truth": gt, "prediction": pred, "is_correct": is_correct,
        "mode": "baseline", "raw_output": result["raw_output"]
    })

    if total % 1 == 0:
        print(f"  Progress: {total} done, {correct}/{total} correct so far...")

elapsed = time.time() - start
print(f"\nBaseline: {correct}/{total} ({correct/total*100:.2f}%) in {elapsed:.1f}s")
print(f"Saved to: {baseline_path}")


## 8. Experiment: Adaptive Skill Generation (Iterative Prompting)


In [ ]:
ITERATIONS = 2  # Number of iterative prompting rounds

# Save directly to Google Drive
skills_path = os.path.join(DRIVE_SAVE_DIR, f"results_color_recognition_adaptive_skills_iter{ITERATIONS}.jsonl")
skills_logger = IncrementalLogger(skills_path)

# TEST MODE: Set to False to run all 76 questions
QUICK_TEST = True
limit = 3 if QUICK_TEST else len(dataset_items)

correct, total = 0, 0
start = time.time()

for item in dataset_items[:limit]:
    if item["idx"] in skills_logger.processed_indices:
        continue

    question = item["question"]
    choices = item["choices"]
    image = item["image"]

    # Iteration 1: Generate initial skill (visually grounded)
    plan = ace_system.generate_skill(question, choices, image=image)
    skill = plan.get("skill", "")
    classification = plan.get("classification", "")

    result = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
    prediction = result["prediction"]
    raw_output = result["raw_output"]

    all_iterations = [{"iter": 1, "skill": skill, "prediction": prediction}]

    # Iterations 2+: Reflect on mistakes and refine skill
    for i in range(2, ITERATIONS + 1):
        refined = ace_system.reflect_and_refine(
            question, choices, skill, prediction, raw_output, image=image
        )
        skill = refined.get("skill", skill)
        reflection = refined.get("reflection", "")

        result = solver.solve(image, question, choices, mode="adaptive_skills", skill=skill)
        prediction = result["prediction"]
        raw_output = result["raw_output"]

        all_iterations.append({"iter": i, "skill": skill, "prediction": prediction, "reflection": reflection})

    gt = item["answer"]
    is_correct = (prediction == gt)
    if is_correct: correct += 1
    total += 1

    skills_logger.log_result({
        "idx": item["idx"], "id": item["id"], "task": item["task"],
        "question": question, "choices": choices,
        "ground_truth": gt, "prediction": prediction, "is_correct": is_correct,
        "mode": "adaptive_skills", "classification": classification,
        "skill": skill, "iterations": all_iterations,
        "raw_output": raw_output
    })

    if total % 1 == 0:
        print(f"  Progress: {total} done, {correct}/{total} correct so far...")

elapsed = time.time() - start
print(f"\nAdaptive Skills ({ITERATIONS} iters): {correct}/{total} ({correct/total*100:.2f}%) in {elapsed:.1f}s")
print(f"Saved to: {skills_path}")


## 9. Compare All Results


In [ ]:
# Compare results stored in Google Drive
!python eval_results.py --dir "{DRIVE_SAVE_DIR}"
